# Chord Comparison

A "chord" = multiple(>=3) notes with the same or very close start times
- Block Chords: multiple(>=3) notes with the same start time
- Broken Chords / Arpeggios: multiple(>=3) notes play one after another, in a given order
focus on block chords first. 

Chord comparison can be considered as a set matching problem so that the correctness of a chord can be determined by the overlap between response set and reference set. Additionally, it may require handling partial matches problem (e.g. two out of three chord notes played correctly), consider defining it as imperfect chords. 

In [35]:
import os
import json

cwd = os.getcwd()

cwd = os.getcwd()
dir = os.path.dirname(cwd)
reference_path = os.path.join(dir, "data", "referenceMIDIchords.json")
response_path = os.path.join(dir, "data", "responseMIDIchords.json")

with open(reference_path) as f1:
    ref_chords_data = json.load(f1)

with open(response_path) as f2:
    res_chords_data = json.load(f2)

In [36]:
from evaluation_function.compare_MIDI import compare_performance_ED

chord accuracy metric: https://arxiv.org/pdf/2201.05244

consider chord as unordered set of pitch classes
https://www.researchgate.net/profile/Pierre-Hanna/publication/265984596_A_Survey_Of_Chord_Distances_With_Comparison_For_Chord_Analysis/links/555c4c7808aec5ac2232b158/A-Survey-Of-Chord-Distances-With-Comparison-For-Chord-Analysis.pdf

In [37]:
result = compare_performance_ED(res_chords_data, ref_chords_data)
print(result.feedback_message)

Overview: 
Timing: your overall tempo is within an acceptable range. Good job! The timing is about 1% behind the reference in general while notes are held about 0% shorter than the reference.
There are no pitch errors. Well done!
There are no missing notes. Great!
There are no extra notes. Good job!
Chords: 1/3 correct, 1/3 imperfect (some notes missing or extra), 1/3 completely wrong.

All notes(melody part) played correctly!

Chord Detail:
Chord 2 (expected F major, you played unknown chord): 83% accurate. Missing note(s): F. 
Chord 3 (expected G major, you played A minor): 0% accurate. Missing note(s): D, G, B. Extra note(s) played: C, E, A.


In [38]:
test_path = os.path.join(dir, "data", "test_cases_chords.json")

with open(test_path) as f:
    test_cases = json.load(f)

print(f"Loaded {len(test_cases)} test cases")
for case in test_cases:
    print(f"  - {case['case_id']} ({case['length_category']})")

Loaded 11 test cases
  - perfect_melody (short)
  - perfect_chord (short)
  - imperfect_chord_missing_one_note (short)
  - wrong_chord_no_overlap (short)
  - chord_with_extra_note (short)
  - missing_chord (short)
  - mixed_melody_and_chord_perfect (short)
  - mixed_melody_wrong_pitch_imperfect_chord (short)
  - type_mismatch_note_vs_chord (short)
  - chord_timing_off (short)
  - multiple_chords_mixed_accuracy (long)


In [39]:
# Pick one case by id to inspect closely
case_id = "multiple_chords_mixed_accuracy"
case = next(c for c in test_cases if c["case_id"] == case_id)

print("Purpose:", case["purpose"])
print()

result = compare_performance_ED(case["response"], case["reference"])

print("is_correct:", result.is_correct)
print("stats:", result.stats)
print()
print(result.feedback_message)

Purpose: Two chords: first is perfect C major, second is imperfect A minor (missing E). total_chords_correct = 1, total_chords_imperfect = 1.

is_correct: False
stats: {'pitch_all_aligned_correct': False, 'timing_all_correct': True, 'duration_all_correct': True, 'total_notes_in_reference': 0, 'total_notes_missing': 0, 'total_notes_extra': 0, 'total_notes_wrong_pitch': 0, 'total_notes_wrong_timing': 0, 'total_notes_wrong_duration': 0, 'total_notes_correct': 0, 'timing_scale': 1.0, 'timing_offset': 0.0, 'duration_scale': 1.0, 'total_chords_in_reference': 2, 'total_chords_missing': 0, 'total_chords_extra': 0, 'total_chords_correct': 1, 'total_chords_imperfect': 1, 'total_chords_wrong': 0}

Overview: 
Timing: your overall tempo is within an acceptable range. Good job! The timing is about 0% ahead of the reference in general while notes are held about 0% shorter than the reference.
There are no pitch errors. Well done!
There are no missing notes. Great!
There are no extra notes. Good job!
C

In [41]:
for case in test_cases:
    result = compare_performance_ED(case["response"], case["reference"])
    s = result.stats
    print(
        f"[{case['case_id']:40s}] "
        f"is_correct={result.is_correct!s:5} "
        f"pitch_all={s['pitch_all_aligned_correct']!s:5} "
        f"timing_all={s['timing_all_correct']!s:5} "
        f"duration_all={s['duration_all_correct']!s:5} "
        f"notes_ref={s['total_notes_in_reference']} "
        f"missing={s['total_notes_missing']} "
        f"extra={s['total_notes_extra']} "
        f"wrong_pitch={s['total_notes_wrong_pitch']} "
        f"wrong_timing={s['total_notes_wrong_timing']} "
        f"wrong_dur={s['total_notes_wrong_duration']} "
        f"notes_correct={s['total_notes_correct']} "
        f"chords_ref={s['total_chords_in_reference']} "
        f"chords_missing={s['total_chords_missing']} "
        f"chords_extra={s['total_chords_extra']} "
        f"chords_correct={s['total_chords_correct']} "
        f"chords_imperfect={s['total_chords_imperfect']} "
        f"chords_wrong={s['total_chords_wrong']} "
        f"timing_scale={s['timing_scale']:.2f} "
        f"duration_scale={s['duration_scale']:.2f}"
    )

[perfect_melody                          ] is_correct=True  pitch_all=True  timing_all=True  duration_all=True  notes_ref=4 missing=0 extra=0 wrong_pitch=0 wrong_timing=0 wrong_dur=0 notes_correct=4 chords_ref=0 chords_missing=0 chords_extra=0 chords_correct=0 chords_imperfect=0 chords_wrong=0 timing_scale=1.00 duration_scale=1.00
[perfect_chord                           ] is_correct=True  pitch_all=True  timing_all=True  duration_all=True  notes_ref=0 missing=0 extra=0 wrong_pitch=0 wrong_timing=0 wrong_dur=0 notes_correct=0 chords_ref=1 chords_missing=0 chords_extra=0 chords_correct=1 chords_imperfect=0 chords_wrong=0 timing_scale=1.00 duration_scale=1.00
[imperfect_chord_missing_one_note        ] is_correct=False pitch_all=False timing_all=True  duration_all=True  notes_ref=0 missing=0 extra=0 wrong_pitch=0 wrong_timing=0 wrong_dur=0 notes_correct=0 chords_ref=1 chords_missing=0 chords_extra=0 chords_correct=0 chords_imperfect=1 chords_wrong=0 timing_scale=1.00 duration_scale=1.00
[